# Phase 2 Legal NER · Small-model method comparison

This is a separate experiment notebook. It reuses the existing pinned Phase 2 data loader, split, tokenizer, character-edge candidates, `SpanNER` head, and strict exact-span evaluator. Each method writes to a method/config/run-ID-specific folder with resumable full model, optimizer, scheduler, AMP scaler, RNG, and progress state.

Methods: **full**, **partial** layer fine-tuning, **LoRMA+** multiplicative adapters, and **KD** from a completed trained Phase 2 teacher into an existing smaller Phase 1 student. KD reads training step, labels, tokenizer identity, and the complete span head from the teacher checkpoint itself; weak validation quality is reported as a warning. No scores are supplied by this notebook; scores are computed only after an actual run.

In [16]:
# 1 · Choose method and architecture preset; checkpoints are discovered after Drive mount
from pathlib import Path
ACTION = 'Train'  # Inspect | Train | Predict
METHOD = 'full'  # full | partial | lorma_plus | kd
RUN_ID = 'mha-prenorm-full-001'  # Change per independent run; reused only to resume that run.
COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
TRAIN_DIR = COURSE_ROOT / 'Phase_2_LNER/Train'
TEST_DIR = COURSE_ROOT / 'Phase_2_LNER/Test'
RUNS_ROOT = COURSE_ROOT / 'runs/phase2_lner_multimethod'
BUNDLE_ZIP = COURSE_ROOT / 'phase2_lner_multimethod_bundle.zip'
MHA_PRE_SNAPSHOT = (COURSE_ROOT / 'runs/mha_pre/run-001/legal-token-stages-v1/'
                    'run-001-milestone-60762-aa4db459')

# Preset architecture values were read from copied, existing snapshot manifests.
# At runtime the notebook discovers matching payload checkpoints after Drive mount.
PRESET_NAME = 'MHA-prenorm'  # GQA-prenorm | MHA-prenorm | MHA-postnorm | MLA-prenorm
MODEL_PRESETS = {
 'GQA-prenorm': {'vocab_size':32768,'d_model':512,'layers':12,'heads':8,'kv_heads':2,
  'ffn_hidden':2048,'context_length':2048,'attention':'gqa','norm_placement':'pre',
  'tie_embeddings':True,'rope_base':10000,'norm_epsilon':1e-6,'init_std':.02},
 'MHA-postnorm': {'vocab_size':32768,'d_model':512,'layers':12,'heads':8,'kv_heads':8,
  'ffn_hidden':2048,'context_length':2048,'attention':'mha','norm_placement':'post',
  'tie_embeddings':True,'rope_base':10000,'norm_epsilon':1e-6,'init_std':.02},
 'MHA-prenorm': {'vocab_size':32768,'d_model':512,'layers':12,'heads':8,'kv_heads':8,
  'ffn_hidden':2048,'context_length':2048,'attention':'mha','norm_placement':'pre',
  'tie_embeddings':True,'rope_base':10000,'norm_epsilon':1e-6,'init_std':.02},
 'MLA-prenorm': {'vocab_size':32768,'d_model':512,'layers':12,'heads':8,'kv_heads':2,
  'ffn_hidden':2048,'context_length':2048,'attention':'mla','norm_placement':'pre',
  'tie_embeddings':True,'rope_base':10000,'norm_epsilon':1e-6,'init_std':.02,
  'mla_latent':256,'mla_query_rank':256,'mla_rope_dim':32}}
if PRESET_NAME not in MODEL_PRESETS: raise ValueError(f'Unknown preset: {PRESET_NAME}')
MODEL_CONFIG = MODEL_PRESETS[PRESET_NAME]
PARTIAL_LAYERS = (10, 11)  # partial: explicit zero-based decoder block indices
LORMA_TARGETS = ('output',)  # same square target exists in MHA, GQA, and MLA
LORMA_RANK, LORMA_ALPHA = 4, 8

# KD uses a Phase 1 GQA student snapshot and the explicitly selected larger MHA Phase 2 teacher.
# The healthy MHA-prenorm Phase 2 run is the default; select postnorm only deliberately.
KD_STUDENT_PRESET = 'GQA-prenorm'
KD_TEACHER_PRESET = 'MHA-prenorm'  # MHA-prenorm | MHA-postnorm

MAX_TOKENS, STRIDE, MAX_SPAN_TOKENS = 512, 384, 64
SEED, VALIDATION_FRACTION = 17, .15
TRAIN_NEGATIVES, INFERENCE_CHUNK = 8192, 8192
LEARNING_RATE, WEIGHT_DECAY = 2e-5, .01
assert ACTION in ('Inspect','Train','Predict')
assert METHOD in ('full','partial','lorma_plus','kd')
if ACTION == 'Inspect': print('Inspect mode: no model/checkpoint is loaded.')

In [17]:
# 2 · Mount Drive and load this distinct helper bundle
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, hashlib, json, importlib.util, subprocess, zipfile
LOCAL = Path('/content/phase2-lner-multimethod')
if LOCAL.exists(): shutil.rmtree(LOCAL)
if not BUNDLE_ZIP.is_file(): raise FileNotFoundError(f'Upload the separate helper bundle first: {BUNDLE_ZIP}')
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive: archive.extractall(LOCAL)
sys.path.insert(0, str(LOCAL))
missing=[]
if importlib.util.find_spec('tokenizers') is None: missing.append('tokenizers==0.23.1')
if importlib.util.find_spec('huggingface_hub') is None: missing.append('huggingface_hub')
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from tokenizers import Tokenizer
import torch
from legal_ner import *
from legal_scaffold.models import ModelConfig, CausalDecoder
from training_methods import (restore_backbone, configure_mode, distillation_step,
    shared_span_distillation_loss, resolve_snapshot_checkpoint,
    load_trained_teacher_checkpoint)
from legal_ner import (save_training_checkpoint, resume_training_checkpoint,
                       load_inference_checkpoint)
TOKENIZER_PATH = LOCAL/'tokenizer/tokenizer.json'
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
if hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest() != TOKENIZER_SHA256:
    raise ValueError('The copied frozen tokenizer differs from the existing Phase 2 tokenizer.')
TOKENIZER = Tokenizer.from_file(str(TOKENIZER_PATH))
print('Separate helper and existing frozen tokenizer loaded.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Separate helper and existing frozen tokenizer loaded.


In [18]:
# 3 · Load the pinned dataset pairs and inspect actual annotations
from google.colab import userdata
training_files = ensure_training_files(COURSE_ROOT, userdata.get('HF_TOKEN'))
files = sorted(p for p in TRAIN_DIR.rglob('*') if p.is_file())
load_labels(next(p for p in files if p.name.lower() == 'labels.txt'))
text_files = {p.name[:-10]:p for p in files if p.name.endswith('-text.json')}
gold_files = {p.name[:-10]:p for p in files if p.name.endswith('-gold.json')}
if not text_files or set(text_files) != set(gold_files):
    raise ValueError(f'Expected paired official fold files; text={sorted(text_files)}, gold={sorted(gold_files)}')
rows = []
for fold in sorted(text_files): rows.extend(load_fold_pair(text_files[fold], gold_files[fold]))
if len({r['id'] for r in rows}) != len(rows): raise ValueError('document ID repeats across official folds')
report = audit(rows)
print(json.dumps(report, ensure_ascii=False, indent=2))
print('The audited official data are the source of truth for offsets and overlap.')

{
  "documents": 70,
  "entities": 8393,
  "label_counts": {
    "APP": 407,
    "RESP": 381,
    "A.COUNSEL": 195,
    "R.COUNSEL": 184,
    "JUDGE": 245,
    "WIT": 187,
    "AUTH": 1003,
    "COURT": 1013,
    "STAT": 1260,
    "PREC": 1183,
    "DATE": 1575,
    "CASENO": 760
  },
  "unicode_documents": 29,
  "overlapping_pairs": 0,
  "nested_pairs": 0,
  "document_chars": {
    "0": 10443,
    "0.5": 26600,
    "0.9": 48800,
    "0.99": 106947,
    "1": 114044
  },
  "span_chars": {
    "0": 1,
    "0.5": 13,
    "0.9": 33,
    "0.99": 85,
    "1": 233
  },
  "substring_evidence_entities": 8393,
  "offset_convention": "verified on every entity carrying entity_text: Python Unicode code points, start inclusive, end exclusive"
}
The audited official data are the source of truth for offsets and overlap.


In [19]:
# 4 · The one deterministic split, order, windows, and span-support audit
train_rows, val_rows = split_documents(rows, VALIDATION_FRACTION, SEED)
assert {r['id'] for r in train_rows}.isdisjoint({r['id'] for r in val_rows})
train_windows = [w for r in train_rows for w in tokenize_windows(r,TOKENIZER,MAX_TOKENS,STRIDE)]
val_windows = [w for r in val_rows for w in tokenize_windows(r,TOKENIZER,MAX_TOKENS,STRIDE)]
covered = {(w['document_id'],a,b,l) for w in train_windows+val_windows for a,b,l in w['entities']}
missing = [(r['id'],*e) for r in rows for e in r['entities'] if (r['id'],*e) not in covered]
if missing: raise ValueError(f'Gold spans uncovered by shared windows: {missing[:5]}')
representable = set()
for w in train_windows+val_windows:
    for a,b,label in w['entities']:
        ij = token_span_for_chars(w,a,b)
        if ij is not None and ij[1]-ij[0]+1 <= MAX_SPAN_TOKENS:
            representable.add((w['document_id'],a,b,label))
missing = [(r['id'],*e) for r in rows for e in r['entities'] if (r['id'],*e) not in representable]
if missing: raise ValueError(f'Gold spans unrepresentable by shared candidate policy: {missing[:5]}')
print({'train_docs':len(train_rows),'val_docs':len(val_rows),
       'train_windows':len(train_windows),'val_windows':len(val_windows),
       'split_seed':SEED,'max_span_tokens':MAX_SPAN_TOKENS})

{'train_docs': 64, 'val_docs': 6, 'train_windows': 1282, 'val_windows': 111, 'split_seed': 17, 'max_span_tokens': 64}


## Method and checkpoint setup

The preset configs come from copied snapshot manifests. After Drive is mounted, the resolver searches actual snapshot directories, reads `snapshot.json` and `state_inventory.model`, checks config and tokenizer metadata, and requires a real state payload. For KD it also finds a trained `latest.pt` whose embedded checkpoint metadata includes positive steps and a complete NER head. Predict rebuilds the method-specific module tree before restoring the trained checkpoint.

In [20]:
# 5 · Resolve existing snapshot payloads and prepare the selected method
MODEL_CFG=ModelConfig(**MODEL_PRESETS[PRESET_NAME]).validate()
STUDENT_CFG=ModelConfig(**MODEL_PRESETS[KD_STUDENT_PRESET]).validate()
TEACHER_CFG=ModelConfig(**MODEL_PRESETS[KD_TEACHER_PRESET]).validate()
if ACTION=='Inspect':
    model=teacher=trainable_mask=None; RUN_ROOT=RUNS_ROOT/METHOD/RUN_ID; IDENTITY=None
    print('Inspect: weights are not loaded. Snapshot payloads discovered from Drive:')
    for preset_name,preset_config in MODEL_PRESETS.items():
        try:
            payload,manifest=resolve_snapshot_checkpoint(COURSE_ROOT/'runs',
                ModelConfig(**preset_config),stages=('legal_continued','wikitext_only'),
                tokenizer_sha256=TOKENIZER_SHA256)
            print(preset_name,'->',payload,'stage=',manifest.get('stage'),'step=',manifest.get('progress',{}).get('step'))
        except FileNotFoundError:
            print(preset_name,'-> no matching Drive snapshot payload found')
else:
    if METHOD=='kd':
        student_checkpoint,student_manifest=resolve_snapshot_checkpoint(
            COURSE_ROOT/'runs',STUDENT_CFG,stages=('wikitext_only','legal_continued'),
            tokenizer_sha256=TOKENIZER_SHA256)
        model=SpanNER(restore_backbone(STUDENT_CFG,student_checkpoint),STUDENT_CFG.d_model)
        teacher=SpanNER(CausalDecoder(TEACHER_CFG),TEACHER_CFG.d_model)
        candidates=sorted((COURSE_ROOT/'runs').rglob('latest.pt'),key=lambda p:p.stat().st_mtime,reverse=True)
        teacher_checkpoint=None; teacher_progress=None; teacher_identity=None
        for candidate in candidates:
            try:
                progress,identity=load_trained_teacher_checkpoint(
                    candidate,teacher,tokenizer_sha256=TOKENIZER_SHA256,labels=LABELS,
                    expected_model_config=TEACHER_CFG,
                    expected_windows=len(train_windows))
                if sum(p.numel() for p in teacher.backbone.parameters()) <= sum(p.numel() for p in model.backbone.parameters()):
                    continue
                teacher_checkpoint=candidate; teacher_progress=progress; teacher_identity=identity; break
            except (ValueError,RuntimeError,KeyError,TypeError,OSError,EOFError):
                continue
        if teacher_checkpoint is None:
            raise FileNotFoundError(f'No completed trained Phase 2 teacher matching {KD_TEACHER_PRESET}, labels, tokenizer, and shared training windows, with a complete span head and larger backbone, was found under Drive runs. Train the explicitly selected {KD_TEACHER_PRESET} teacher first.')
        MODEL_CFG=STUDENT_CFG; config_record=STUDENT_CFG.__dict__
        IDENTITY={'schema_version':2,'method':METHOD,'run_id':RUN_ID,'model_config':config_record,
          'labels':list(LABELS),'tokenizer_sha256':TOKENIZER_SHA256,'max_tokens':MAX_TOKENS,
          'stride':STRIDE,'max_span_tokens':MAX_SPAN_TOKENS,'seed':SEED,
          'validation_fraction':VALIDATION_FRACTION,'student_checkpoint':str(student_checkpoint),
          'student_snapshot_sha256':hashlib.sha256((student_checkpoint.parent/'snapshot.json').read_bytes()).hexdigest(),
          'teacher_checkpoint':str(teacher_checkpoint),
          'teacher_identity':teacher_identity}
        teacher.eval()
    else:
        if PRESET_NAME=='MHA-prenorm':
            from legal_scaffold.snapshots import safe_file
            manifest_path=MHA_PRE_SNAPSHOT/'snapshot.json'
            if not manifest_path.is_file(): raise FileNotFoundError(
                f'MHA-prenorm manifest missing or inaccessible: {manifest_path}. '
                'Check that Drive is mounted and this account can read the mha_pre snapshot folder.')
            base_manifest=json.loads(manifest_path.read_text())
            saved_config=ModelConfig(**base_manifest['identity']['model']).validate().__dict__
            if saved_config != MODEL_CFG.__dict__:
                raise ValueError('The explicit MHA-prenorm snapshot config differs from the selected preset.')
            saved_tokenizer=base_manifest['identity']['tokenizer_identity']['tokenizer_sha256']
            if saved_tokenizer != TOKENIZER_SHA256:
                raise ValueError('The explicit MHA-prenorm snapshot tokenizer differs from the frozen tokenizer.')
            relative_state=base_manifest['state_inventory']['model']
            safe_state=safe_file(relative_state)
            base_checkpoint=MHA_PRE_SNAPSHOT/safe_state
            if (base_manifest.get('stage')!='legal_continued' or
                    relative_state not in base_manifest.get('artifacts',{}) or
                    not base_checkpoint.is_file()):
                raise FileNotFoundError(
                    f'MHA-prenorm state payload missing, inaccessible, or not inventoried: {base_checkpoint}. '
                    'Check Drive mount access to the mha_pre snapshot folder.')
        else:
            base_checkpoint,base_manifest=resolve_snapshot_checkpoint(
                COURSE_ROOT/'runs',MODEL_CFG,stages=('legal_continued','wikitext_only'),
                tokenizer_sha256=TOKENIZER_SHA256)
        model=SpanNER(restore_backbone(MODEL_CFG,base_checkpoint),MODEL_CFG.d_model)
        teacher=None; config_record=MODEL_CFG.__dict__
        IDENTITY={'schema_version':2,'method':METHOD,'run_id':RUN_ID,'model_config':config_record,
          'labels':list(LABELS),'tokenizer_sha256':TOKENIZER_SHA256,'max_tokens':MAX_TOKENS,
          'stride':STRIDE,'max_span_tokens':MAX_SPAN_TOKENS,'seed':SEED,
          'validation_fraction':VALIDATION_FRACTION,'base_checkpoint':str(base_checkpoint),
          'base_snapshot_sha256':hashlib.sha256((base_checkpoint.parent/'snapshot.json').read_bytes()).hexdigest(),
          'partial_layers':list(PARTIAL_LAYERS) if METHOD=='partial' else [],
          'lorma':{'targets':list(LORMA_TARGETS),'rank':LORMA_RANK,'alpha':LORMA_ALPHA} if METHOD=='lorma_plus' else None}
    IDENTITY['dataset_repo']=DATASET_REPO; IDENTITY['dataset_revision']=DATASET_REVISION
    fingerprint=hashlib.sha256(json.dumps(config_record,sort_keys=True).encode()).hexdigest()[:10]
    RUN_ROOT=RUNS_ROOT/METHOD/f'{RUN_ID}-{fingerprint}'
    trainable_mask=configure_mode(model,METHOD,layers=PARTIAL_LAYERS,
        lorma_targets=LORMA_TARGETS,rank=LORMA_RANK,alpha=LORMA_ALPHA)
    print('Resolved architecture:',config_record)
    if METHOD=='kd':
        print('Resolved GQA student snapshot:',student_checkpoint,'stage:',student_manifest.get('stage'),
              'progress:',student_manifest.get('progress'))
        if student_manifest.get('stage')!='wikitext_only':
            print('WARNING: copied manifests had no GQA wikitext-only snapshot; selected the matching legal_continued GQA snapshot.')
    if METHOD=='kd': print('Resolved trained Phase2 teacher:',teacher_checkpoint,'progress:',teacher_progress)
    else: print('Resolved base snapshot:',base_checkpoint)
    print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad),
          '/',sum(p.numel() for p in model.parameters()),'run folder:',RUN_ROOT)

Resolved architecture: {'vocab_size': 32768, 'd_model': 512, 'layers': 12, 'heads': 8, 'kv_heads': 8, 'ffn_hidden': 2048, 'context_length': 2048, 'attention': 'mha', 'norm_placement': 'pre', 'tie_embeddings': True, 'rope_base': 10000, 'norm_epsilon': 1e-06, 'init_std': 0.02, 'mla_latent': None, 'mla_query_rank': None, 'mla_rope_dim': None}
Resolved base snapshot: /content/drive/MyDrive/legal-llm-course/runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459/state.pt
Trainable parameters: 67654157 / 67654157 run folder: /content/drive/MyDrive/legal-llm-course/runs/phase2_lner_multimethod/full/mha-prenorm-full-001-6397e0631f


In [21]:
# 6 · Shared candidate batches and the unchanged strict span evaluator
import torch.nn.functional as F
AMP_DTYPE=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
def span_batch(window, device='cpu'):
    spans = candidate_spans(window, MAX_SPAN_TOKENS)
    positives = [x for x in spans if x[4] != 0]
    negatives = [x for x in spans if x[4] == 0]
    if len(negatives) > TRAIN_NEGATIVES:
        stride = len(negatives) / TRAIN_NEGATIVES
        negatives = [negatives[int(k*stride)] for k in range(TRAIN_NEGATIVES)]
    spans = positives + negatives
    input_ids = torch.tensor([window['input_ids']], dtype=torch.long, device=device)
    span_indices = torch.tensor([[(x[0],x[1],x[5],x[6]) for x in spans]],dtype=torch.long,device=device)
    labels = torch.tensor([x[4] for x in spans],dtype=torch.long,device=device)
    return input_ids, span_indices, labels, spans

@torch.inference_mode()
def predict_windows(model, windows, device):
    model.eval().to(device); output=[]
    for window in windows:
        spans = candidate_spans(window, MAX_SPAN_TOKENS)
        ids = torch.tensor([window['input_ids']],dtype=torch.long,device=device)
        amp = torch.autocast('cuda',dtype=AMP_DTYPE) if device=='cuda' else torch.autocast('cpu',enabled=False)
        with amp:
            hidden=model.hidden(ids)
            for start in range(0,len(spans),INFERENCE_CHUNK):
                part = spans[start:start+INFERENCE_CHUNK]
                if not part: continue
                indices = torch.tensor([[(x[0],x[1],x[5],x[6]) for x in part]],dtype=torch.long,device=device)
                probabilities = model.score_hidden(hidden,indices).softmax(-1)[0]
                scores, classes = probabilities[:,1:].max(-1); classes += 1
                for span,score,label_id in zip(part,scores.tolist(),classes.tolist()):
                    output.append((window['document_id'],span[2],span[3],LABELS[label_id-1],score))
    return merge_predictions(output)

def macro_f1(rows, predictions):
    gold={(r['id'],a,b,label) for r in rows for a,b,label in r['entities']}
    pred={(d,a,b,label) for d,a,b,label,_ in predictions}; scores={}
    for label in LABELS:
        g={x for x in gold if x[3]==label}; p={x for x in pred if x[3]==label}; tp=len(g&p)
        precision=tp/len(p) if p else 0; recall=tp/len(g) if g else 0
        scores[label]=2*precision*recall/(precision+recall) if precision+recall else 0
    return scores,sum(scores.values())/len(LABELS)
print('Common span-batch and exact-span evaluation functions are ready.')
if METHOD=='kd' and ACTION=='Train':
    device='cuda' if torch.cuda.is_available() else 'cpu'
    teacher_predictions=predict_windows(teacher,val_windows,device)
    _,teacher_macro=macro_f1(val_rows,teacher_predictions)
    print('Teacher Phase 2 strict validation macro F1:',teacher_macro)
    if teacher_macro < .10: print('WARNING: teacher validation quality is weak; KD will proceed, inspect teacher predictions and labels.')

Common span-batch and exact-span evaluation functions are ready.


In [22]:
# 7 · Train or resume this method; checkpoint includes adapter/head and optimizer state
from torch.optim import AdamW
import time
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = RUN_ROOT/'latest.pt'
if ACTION == 'Train':
    if device != 'cuda': raise RuntimeError('Train requires CUDA; no CPU fallback is used for production runs.')
    AMP_DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler=torch.amp.GradScaler('cuda',enabled=(AMP_DTYPE==torch.float16))
    RUN_ROOT.mkdir(parents=True,exist_ok=True)
    model.to(device).train()
    if teacher is not None: teacher.to(device).eval()
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                      lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,len(train_windows)))
    progress={'epoch':0,'next_window':0,'step':0}
    if checkpoint.exists():
        progress=resume_training_checkpoint(checkpoint,model,optimizer,scheduler,IDENTITY,scaler=scaler)
    # Older checkpoints tracked step as attempted windows; retain that count as a
    # compatibility fallback while new checkpoints record both counts explicitly.
    progress.setdefault('consumed_windows',progress.get('next_window',0))
    progress.setdefault('successful_updates',progress.get('step',0))
    progress['step']=progress['successful_updates']
    train_started=time.monotonic()
    for index in range(progress['next_window'],len(train_windows)):
        optimizer.zero_grad(set_to_none=True)
        input_ids,span_indices,gold_labels,_=span_batch(train_windows[index],device)
        with torch.autocast('cuda',dtype=AMP_DTYPE):
            logits=model(input_ids,span_indices).reshape(-1,len(LABELS)+1)
            if METHOD=='kd':
                with torch.no_grad(): teacher_logits=teacher(input_ids,span_indices)
                loss=shared_span_distillation_loss(logits,teacher_logits,gold_labels,
                                                    temperature=2.0,hard_weight=.5)
            else:
                loss=F.cross_entropy(logits.float(),gold_labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
        previous_scale=scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        optimizer_updated=(not scaler.is_enabled()) or scaler.get_scale() >= previous_scale
        if optimizer_updated: scheduler.step()
        loss_value=float(loss.detach())
        consumed_windows=index+1
        successful_updates=progress['successful_updates']+int(optimizer_updated)
        progress={'epoch':0,'next_window':consumed_windows,'consumed_windows':consumed_windows,
                  'successful_updates':successful_updates,'step':successful_updates}
        elapsed=time.monotonic()-train_started
        print(f'window {consumed_windows}/{len(train_windows)} updates={successful_updates} '
              f'loss={loss_value:.6f} scale={scaler.get_scale():g} elapsed={elapsed:.1f}s',flush=True)
        if consumed_windows%25==0:
            save_training_checkpoint(checkpoint,model,optimizer,scheduler,progress,IDENTITY,scaler=scaler)
    save_training_checkpoint(checkpoint,model,optimizer,scheduler,progress,IDENTITY,scaler=scaler)
    print('Training pass saved:',progress,'checkpoint:',checkpoint)
else:
    print('Training skipped.')

window 1/1282 updates=1 loss=2.960086 scale=1 elapsed=1.9s
window 2/1282 updates=2 loss=2.488817 scale=1 elapsed=2.3s
window 3/1282 updates=3 loss=2.545162 scale=1 elapsed=2.6s
window 4/1282 updates=4 loss=2.259870 scale=1 elapsed=3.0s
window 5/1282 updates=5 loss=1.893937 scale=1 elapsed=3.3s
window 6/1282 updates=6 loss=1.853674 scale=1 elapsed=3.6s
window 7/1282 updates=7 loss=1.967408 scale=1 elapsed=4.0s
window 8/1282 updates=8 loss=2.022112 scale=1 elapsed=4.3s
window 9/1282 updates=9 loss=1.956056 scale=1 elapsed=4.7s
window 10/1282 updates=10 loss=1.710286 scale=1 elapsed=5.0s
window 11/1282 updates=11 loss=1.859657 scale=1 elapsed=5.3s
window 12/1282 updates=12 loss=1.919321 scale=1 elapsed=5.6s
window 13/1282 updates=13 loss=1.826978 scale=1 elapsed=5.8s
window 14/1282 updates=14 loss=1.935537 scale=1 elapsed=6.1s
window 15/1282 updates=15 loss=1.851132 scale=1 elapsed=6.6s
window 16/1282 updates=16 loss=1.534532 scale=1 elapsed=7.1s
window 17/1282 updates=17 loss=1.603382 sc

In [23]:
# 8 · Predict restores the trained model state before validation or test inference
if ACTION == 'Predict':
    if not checkpoint.is_file(): raise FileNotFoundError(f'Trained checkpoint missing: {checkpoint}')
    # This is the same trained-checkpoint restoration fix used by the v4 baseline:
    # construct the exact method/model shape first, then strictly restore `model`.
    progress = load_inference_checkpoint(checkpoint,model,IDENTITY)
    if not isinstance(progress,dict) or int(progress.get('step',0)) < 1:
        raise ValueError('Predict requires a trained checkpoint with at least one optimizer step.')
    print('Trained model/head/adapters restored:',progress)
    val_predictions=predict_windows(model,val_windows,device)
    per_label,macro=macro_f1(val_rows,val_predictions)
    print('Strict exact-span macro F1:',macro); print(per_label)
    test_files=sorted(p for p in TEST_DIR.rglob('*') if p.suffix.lower() in ('.json','.jsonl') and 'gold' not in p.name.lower())
    test_rows=[]
    for path in test_files: test_rows.extend(load_records(path))
    test_windows=[w for row in test_rows for w in tokenize_windows(row,TOKENIZER,MAX_TOKENS,STRIDE)]
    predictions=predict_windows(model,test_windows,device)
    texts={row['id']:row['text'] for row in test_rows}
    pred_path,zip_path=export_prediction([r['id'] for r in test_rows],predictions,texts,RUN_ROOT/'submission')
    print('Predictions:',pred_path,zip_path)
else:
    print('Predict skipped; no model scores are claimed.')

Predict skipped; no model scores are claimed.


## Run instructions

1. Put the companion bundle at `phase2_lner_multimethod_bundle.zip` in the course Drive folder.
2. Choose `ACTION`, `METHOD`, and `PRESET_NAME`, then run cells in order. Checkpoint directories and payloads are resolved from Drive snapshot manifests. For KD, `KD_TEACHER_PRESET` defaults to MHA-prenorm; its saved checkpoint must embed that exact architecture config. Change it to MHA-postnorm only deliberately, then train that teacher before selecting `METHOD='kd'`.
3. Keep `RUN_ID` to resume the same method/config. Change it for a fresh run. Each method has its own folder. Training requires CUDA; the notebook selects BF16 when supported and FP16 with a saved GradScaler otherwise.

Synthetic CPU train/save/fresh-load/predict checks live beside the notebook in `test_multimethod_e2e.py`. They make no claim about real-data quality.

In [24]:
progress = load_inference_checkpoint(RUN_ROOT/'latest.pt', model, IDENTITY)
val_predictions = predict_windows(model, val_windows, device)
per_label, macro = macro_f1(val_rows, val_predictions)
print('Strict exact-span macro F1:', macro, per_label)

Strict exact-span macro F1: 0.023084426655285043 {'APP': 0, 'RESP': 0, 'A.COUNSEL': 0, 'R.COUNSEL': 0, 'JUDGE': 0, 'WIT': 0, 'AUTH': 0, 'COURT': 0.0416156670746634, 'STAT': 0.04040404040404041, 'PREC': 0, 'DATE': 0.1949934123847167, 'CASENO': 0}
